# Political Intimidation and Voter Turnout - Analysis Notebook
## Author : Mbah Christiana Favour
### Data : Afrobarometer Round 8 Survey, Nigeria
#### Date : 2026

### Import all Library necessary for this analysis

In [1]:
# This installs the library needed to read the SPSS file
# You only need to run this once ever
!pip install pyreadstat

In [2]:
#Importing all the relevant python libraries needed for this project
import pandas as pd #For data handling
import numpy as np #For numerical operations
import pyreadstat  #Allows python read SPSS files
import statsmodels.formula.api as smf #For running logistic regression
import matplotlib.pyplot as plt  #For basic visualization
import seaborn as sns  #For prettier charts
import scipy.stats as stats #For additional statistical tests 

#This line makes your charts appear inside the notebook
%matplotlib inline

print('All libraries imported successfully')

All libraries imported successfully


### Read the cleaned and recoded SPSS file into python

In [3]:
#read cleaned SPSS file into Python
df, meta = pyreadstat.read_sav(
    r'C:\Users\HP\Documents\Thesis Analysis\Afrobarometer_R8.sav'
)

### Double check to confirm the data is loaded correctly

In [4]:
#Checking if the data is loaded correctly
df.head()


,RESPNO,URBRUR,REGION,EA_SVC_A,EA_SVC_B,EA_SVC_C,EA_SVC_D,EA_SVC_E,EA_FAC_A,EA_FAC_B,...,Q119_7,withinwt_ea,withinwt_hh,Age_group,Geopolitical_zone,North_south_region,Religion,Voters,Fear_of_political_intimidation,Witnessed_Political_intimidation
0,NIG0001,1.0,636.0,1.0,1.0,1.0,1.0,1.0,0.0,1.0,...,9996.0,1.268041,2.415317,2.0,4.0,2.0,1.0,0.0,1.0,1.0
1,NIG0002,2.0,634.0,1.0,9.0,0.0,1.0,1.0,0.0,1.0,...,9996.0,1.055457,0.469092,2.0,1.0,1.0,1.0,1.0,1.0,1.0
2,NIG0003,2.0,634.0,1.0,9.0,0.0,1.0,1.0,0.0,1.0,...,9996.0,1.055457,1.876368,3.0,1.0,1.0,2.0,1.0,0.0,0.0
3,NIG0004,2.0,634.0,1.0,9.0,0.0,1.0,1.0,0.0,1.0,...,634.0,1.055457,0.938184,5.0,1.0,1.0,1.0,1.0,1.0,0.0
4,NIG0005,2.0,634.0,1.0,9.0,0.0,1.0,1.0,0.0,1.0,...,9996.0,1.055457,1.407276,1.0,1.0,1.0,1.0,0.0,1.0,0.0


### Count total rows and columns in the dataframe

In [5]:
#Use the shape to determine the Total rows and columns in the dataset
print('Number of Respondents:', df.shape[0])
print('Number of variables:', df.shape[1])

Number of Respondents: 1599
Number of variables: 396


### Confirm the Key variables

In [6]:
#Check key variables

#Check Voter Turnout
print('Voter Turnout Values:')
print(df['Voters'].value_counts())

print('\nPolitical Intimidation Values:')
print(df['Witnessed_Political_intimidation'].value_counts())

print('\nFear of Intimidation values:')
print(df['Fear_of_political_intimidation'].value_counts())

Voter Turnout Values:
Voters
1.0    1247
0.0     351
Name: count, dtype: int64

Political Intimidation Values:
Witnessed_Political_intimidation
0.0    1229
1.0     236
Name: count, dtype: int64

Fear of Intimidation values:
Fear_of_political_intimidation
1.0    797
0.0    794
Name: count, dtype: int64


### Check for missing Values

In [7]:
#Check for missing values
print('Missing values per variable')
print(df[['Voters', 'Witnessed_Political_intimidation', 
          'Fear_of_political_intimidation', 'Age_group', 
          'Geopolitical_zone', 'North_south_region','Gender', 'Religion']].isnull().sum())

Missing values per variable
Voters                                1
Witnessed_Political_intimidation    134
Fear_of_political_intimidation        8
Age_group                             1
Geopolitical_zone                     0
North_south_region                    0
Gender                                0
Religion                             12
dtype: int64


## Data Analysis

### Descriptive Analysis

In [8]:
# Frequencies and percentages for each variable
# This produces Table 1 - sociodemographic characteristics
variables = ['Age_group', 'Gender', 'Religion',
            'Geopolitical_zone', 'North_south_region',
            'Voters', 'Fear_of_political_intimidation',
            'Witnessed_Political_intimidation']
for var in variables:
   print(f"\n--- {var} ---")
   freq = df[var].value_counts()           # count of each category
   pct  = df[var].value_counts(normalize=True) * 100  # percentage
   summary = pd.DataFrame({'Frequency': freq, 'Percentage': pct.round(1)})
   print(summary)


--- Age_group ---
           Frequency  Percentage
Age_group                       
2.0              586        36.7
3.0              415        26.0
4.0              219        13.7
1.0              200        12.5
5.0              178        11.1

--- Gender ---
        Frequency  Percentage
Gender                       
1.0           801        50.1
2.0           798        49.9

--- Religion ---
          Frequency  Percentage
Religion                       
1.0             803        50.6
2.0             784        49.4

--- Geopolitical_zone ---
                   Frequency  Percentage
Geopolitical_zone                       
3.0                      440        27.5
6.0                      312        19.5
1.0                      231        14.4
5.0                      224        14.0
2.0                      224        14.0
4.0                      168        10.5

--- North_south_region ---
                    Frequency  Percentage
North_south_region                       
1

### Inferential Analysis

#### Objective 1 : Demographic factors and Voter turnout

In [9]:
# Binary logistic regression
# voter_turnout is your OUTCOME (dependent variable) - it is 0 or 1
# age_group, gender, religion, north_south are your PREDICTORS
# The formula reads like a sentence:
# "predict voter_turnout using age_group, gender, religion, north_south"
model1 = smf.logit(
   "Voters ~ C(Age_group) + C(Gender) + C(Religion) + C(Geopolitical_zone)",
   data=df
).fit()
# C() tells Python these are categorical variables, not numbers
# Print the full results
print(model1.summary())

Optimization terminated successfully.
         Current function value: 0.490462
         Iterations 6
                           Logit Regression Results                           
Dep. Variable:                 Voters   No. Observations:                 1585
Model:                          Logit   Df Residuals:                     1573
Method:                           MLE   Df Model:                           11
Date:                Wed, 11 Mar 2026   Pseudo R-squ.:                 0.06818
Time:                        15:13:21   Log-Likelihood:                -777.38
converged:                       True   LL-Null:                       -834.26
Covariance Type:            nonrobust   LLR p-value:                 3.244e-19
                                  coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------------------
Intercept                       0.9470      0.239      3.967      0.000   

In [10]:
# Now extract the Odds Ratios (OR) and Confidence Intervals
# This is what you will report in your table
# Odds Ratio = how much more/less likely someone is to vote
odds_ratios_m1 = np.exp(model1.params)
# Confidence Intervals
conf_int_m1 = np.exp(model1.conf_int())
conf_int_m1.columns = ['Lower 95% CI', 'Upper 95% CI']
# P-values
pvalues_m1 = model1.pvalues
# Combine everything into one clean table
results_m1 = pd.concat([odds_ratios_m1, conf_int_m1, pvalues_m1], axis=1)
results_m1.columns = ['Odds Ratio (OR)', 'Lower 95% CI', 'Upper 95% CI', 'p-value']
results_m1 = results_m1.round(3)
print("\nModel 1 Results - Odds Ratios:")
print(results_m1)


Model 1 Results - Odds Ratios:
                             Odds Ratio (OR)  Lower 95% CI  Upper 95% CI  \
Intercept                              2.578         1.615         4.116   
C(Age_group)[T.2.0]                    1.688         1.174         2.429   
C(Age_group)[T.3.0]                    2.523         1.690         3.768   
C(Age_group)[T.4.0]                    3.860         2.333         6.385   
C(Age_group)[T.5.0]                    3.233         1.922         5.438   
C(Gender)[T.2.0]                       0.540         0.419         0.695   
C(Religion)[T.2.0]                     1.801         1.259         2.575   
C(Geopolitical_zone)[T.2.0]            1.057         0.618         1.809   
C(Geopolitical_zone)[T.3.0]            0.878         0.540         1.426   
C(Geopolitical_zone)[T.4.0]            0.597         0.361         0.987   
C(Geopolitical_zone)[T.5.0]            0.584         0.366         0.931   
C(Geopolitical_zone)[T.6.0]            0.525         0.3

In [11]:
# Calculate Nagelkerke R² (measure of how well your model fits)
# Statsmodels does not give this automatically so we calculate it
def nagelkerke_r2(result, n):
   ll_null  = result.llnull   # log likelihood of empty model
   ll_model = result.llf      # log likelihood of your model
   # Cox and Snell R²
   r2_cs  = 1 - np.exp((2 / n) * (ll_null - ll_model))
   # Maximum possible R²
   r2_max = 1 - np.exp((2 / n) * ll_null)
   # Nagelkerke R²
   return r2_cs / r2_max
n = model1.nobs  # number of observations used in the model
nag_r2_m1 = nagelkerke_r2(model1, n)
print(f"Nagelkerke R² for Model 1: {nag_r2_m1:.3f}")

Nagelkerke R² for Model 1: 0.106


#### Objective 2: Demographic factors and Political Intimidation

In [12]:
# Binary logistic regression
# voter_turnout is your OUTCOME (dependent variable) - it is 0 or 1
# age_group, gender, religion, north_south are your PREDICTORS
# The formula reads like a sentence:
# "predict voter_turnout using age_group, gender, religion, north_south"
model2 = smf.logit(
   "Witnessed_Political_intimidation ~ C(Age_group) + C(Gender) + C(Religion) + C(Geopolitical_zone)",
   data=df
).fit()
# C() tells Python these are categorical variables, not numbers
# Print the full results
print(model2.summary())

Optimization terminated successfully.
         Current function value: 0.356893
         Iterations 7
                                  Logit Regression Results                                  
Dep. Variable:     Witnessed_Political_intimidation   No. Observations:                 1453
Model:                                        Logit   Df Residuals:                     1441
Method:                                         MLE   Df Model:                           11
Date:                              Wed, 11 Mar 2026   Pseudo R-squ.:                  0.1873
Time:                                      15:13:21   Log-Likelihood:                -518.56
converged:                                     True   LL-Null:                       -638.04
Covariance Type:                          nonrobust   LLR p-value:                 5.708e-45
                                  coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------

In [13]:
# Extract Odds Ratios for Model 2
odds_ratios_m2 = np.exp(model2.params)
conf_int_m2    = np.exp(model2.conf_int())
conf_int_m2.columns = ['Lower 95% CI', 'Upper 95% CI']
pvalues_m2     = model2.pvalues
results_m2 = pd.concat([odds_ratios_m2, conf_int_m2, pvalues_m2], axis=1)
results_m2.columns = ['Odds Ratio (OR)', 'Lower 95% CI', 'Upper 95% CI', 'p-value']
results_m2 = results_m2.round(3)
print("\nModel 2 Results:")
print(results_m2)
# Nagelkerke R²
n2 = model2.nobs
print(f"\nNagelkerke R² for Model 2: {nagelkerke_r2(model2, n2):.3f}")


Model 2 Results:
                             Odds Ratio (OR)  Lower 95% CI  Upper 95% CI  \
Intercept                              0.172         0.094         0.316   
C(Age_group)[T.2.0]                    0.709         0.429         1.173   
C(Age_group)[T.3.0]                    0.535         0.313         0.915   
C(Age_group)[T.4.0]                    0.565         0.310         1.030   
C(Age_group)[T.5.0]                    0.870         0.483         1.565   
C(Gender)[T.2.0]                       1.084         0.793         1.482   
C(Religion)[T.2.0]                     1.020         0.622         1.672   
C(Geopolitical_zone)[T.2.0]            0.366         0.165         0.813   
C(Geopolitical_zone)[T.3.0]            0.312         0.153         0.637   
C(Geopolitical_zone)[T.4.0]            5.014         2.826         8.895   
C(Geopolitical_zone)[T.5.0]            6.664         3.875        11.462   
C(Geopolitical_zone)[T.6.0]            1.310         0.748         2.2

#### Objective 3: Fear of political Intimidation and Voter Turnout

In [14]:
# Does fear of intimidation predict whether someone votes?
# Control for demographics at the same time
model3 = smf.logit(
   "Voters ~ C(Fear_of_political_intimidation) +C(Age_group) + C(Gender) + C(Religion) + C(Geopolitical_zone)",
   data=df
).fit()
print(model3.summary())

Optimization terminated successfully.
         Current function value: 0.488832
         Iterations 6
                           Logit Regression Results                           
Dep. Variable:                 Voters   No. Observations:                 1577
Model:                          Logit   Df Residuals:                     1564
Method:                           MLE   Df Model:                           12
Date:                Wed, 11 Mar 2026   Pseudo R-squ.:                 0.06663
Time:                        15:13:22   Log-Likelihood:                -770.89
converged:                       True   LL-Null:                       -825.92
Covariance Type:            nonrobust   LLR p-value:                 5.824e-18
                                               coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------------------------------
Intercept                                    0.9

In [15]:
# Extract results
odds_ratios_m3 = np.exp(model3.params)
conf_int_m3    = np.exp(model3.conf_int())
conf_int_m3.columns = ['Lower 95% CI', 'Upper 95% CI']
pvalues_m3     = model3.pvalues
results_m3 = pd.concat([odds_ratios_m3, conf_int_m3, pvalues_m3], axis=1)
results_m3.columns = ['Odds Ratio (OR)', 'Lower 95% CI', 'Upper 95% CI', 'p-value']
results_m3 = results_m3.round(3)
print("\nModel 3 - Fear of Intimidation and Voter Turnout:")
print(results_m3)
n3 = model3.nobs
print(f"\nNagelkerke R²: {nagelkerke_r2(model3, n3):.3f}")


Model 3 - Fear of Intimidation and Voter Turnout:
                                          Odds Ratio (OR)  Lower 95% CI  \
Intercept                                           2.473         1.507   
C(Fear_of_political_intimidation)[T.1.0]            1.051         0.817   
C(Age_group)[T.2.0]                                 1.686         1.172   
C(Age_group)[T.3.0]                                 2.593         1.734   
C(Age_group)[T.4.0]                                 3.846         2.325   
C(Age_group)[T.5.0]                                 3.327         1.967   
C(Gender)[T.2.0]                                    0.548         0.425   
C(Religion)[T.2.0]                                  1.774         1.236   
C(Geopolitical_zone)[T.2.0]                         1.064         0.621   
C(Geopolitical_zone)[T.3.0]                         0.892         0.549   
C(Geopolitical_zone)[T.4.0]                         0.606         0.365   
C(Geopolitical_zone)[T.5.0]                      

#### Objective 4: Actual Witnessing of Political Intimidation and Voter Turnout

In [ ]:
# Does actually WITNESSING intimidation predict whether someone votes?
model4 = smf.logit(
   "Voters ~ C(Witnessed_Political_intimidation) +C(Age_group) + C(Gender) + C(Religion) + C(Geopolitical_zone)",
   data=df
).fit()
print(model4.summary())

In [ ]:
# Extract results
odds_ratios_m4 = np.exp(model4.params)
conf_int_m4    = np.exp(model4.conf_int())
conf_int_m4.columns = ['Lower 95% CI', 'Upper 95% CI']
pvalues_m4     = model4.pvalues
results_m4 = pd.concat([odds_ratios_m4, conf_int_m4, pvalues_m4], axis=1)
results_m4.columns = ['Odds Ratio (OR)', 'Lower 95% CI', 'Upper 95% CI', 'p-value']
results_m4 = results_m4.round(3)
print("\nModel 4 - Actual Intimidation and Voter Turnout:")
print(results_m4)
n4 = model4.nobs
print(f"\nNagelkerke R²: {nagelkerke_r2(model4, n4):.3f}")

#### Objective 5: North vs South comparison 

In [ ]:
# Split the dataset into two groups - North and South
# Then run separate models for each group
# Assuming North = 1, South = 2 in your variable
df_north = df[df['North_south_region'] == 1].copy()  # only northern respondents
df_south = df[df['North_south_region'] == 2].copy()  # only southern respondents
print("Northern respondents:", len(df_north))
print("Southern respondents:", len(df_south))

In [ ]:
# Logistic regression for the NORTH only
model5_north = smf.logit(
   "Voters ~ +C(Witnessed_Political_intimidation)",
   data=df_north
).fit()
print("=== NORTH MODEL ===")
print(model5_north.summary())
or_north = np.exp(model5_north.params)
ci_north = np.exp(model5_north.conf_int())
ci_north.columns = ['Lower 95% CI', 'Upper 95% CI']
pv_north = model5_north.pvalues
res_north = pd.concat([or_north, ci_north, pv_north], axis=1)
res_north.columns = ['OR', 'Lower CI', 'Upper CI', 'p-value']
print("\nNorth - Odds Ratios:")
print(res_north.round(3))
print(f"Nagelkerke R²: {nagelkerke_r2(model5_north, model5_north.nobs):.3f}")

In [ ]:
# Logistic regression for the SOUTH only
model5_south = smf.logit(
   "Voters ~ +C(Witnessed_Political_intimidation)",
   data=df_south
).fit()
print("=== SOUTH MODEL ===")
print(model5_south.summary())
or_south = np.exp(model5_south.params)
ci_south = np.exp(model5_south.conf_int())
ci_south.columns = ['Lower 95% CI', 'Upper 95% CI']
pv_south = model5_south.pvalues
res_south = pd.concat([or_south, ci_south, pv_south], axis=1)
res_south.columns = ['OR', 'Lower CI', 'Upper CI', 'p-value']
print("\nSouth - Odds Ratios:")
print(res_south.round(3))
print(f"Nagelkerke R²: {nagelkerke_r2(model5_south, model5_south.nobs):.3f}")

#### Nagelkerke_r2 Values 

In [ ]:
def nagelkerke_r2(result):
   n = result.nobs
   ll_null  = result.llnull
   ll_model = result.llf
   r2_cs  = 1 - np.exp((2 / n) * (ll_null - ll_model))
   r2_max = 1 - np.exp((2 / n) * ll_null)
   return round(r2_cs / r2_max, 3)
print("Model 1 Nagelkerke R²:", nagelkerke_r2(model1))
print("Model 2 Nagelkerke R²:", nagelkerke_r2(model2))
print("Model 3 Nagelkerke R²:", nagelkerke_r2(model3))
print("Model 4 Nagelkerke R²:", nagelkerke_r2(model4))
print("Model 5 North Nagelkerke R²:", nagelkerke_r2(model5_north))
print("Model 5 South Nagelkerke R²:", nagelkerke_r2(model5_south))

### Save into a excel file for easy integration into Manuscript

In [ ]:
# Save all results to an Excel file so you can copy them into your manuscript
with pd.ExcelWriter("logistic_regression_results.xlsx") as writer:
   results_m1.to_excel(writer, sheet_name="Model1_Demographics_Turnout")
   results_m2.to_excel(writer, sheet_name="Model2_Demographics_Intimidation")
   results_m3.to_excel(writer, sheet_name="Model3_Fear_Turnout")
   results_m4.to_excel(writer, sheet_name="Model4_Intimidation_Turnout")
   res_north.to_excel(writer, sheet_name="Model5_North")
   res_south.to_excel(writer, sheet_name="Model5_South")
print("Results saved to logistic_regression_results.xlsx")
# The file will appear in the same folder where your notebook is saved